## **Imports**

In [1]:
from dotenv import load_dotenv
import os
import json
from datetime import datetime

from langchain_openai import ChatOpenAI

from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

c:\Users\ShoaibMohdKhan\Desktop\rag-chatbot-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## **Load Environment**

In [2]:
load_dotenv()

api_key = os.getenv("OPENROUTER_API_KEY")

print("API Key Loaded:", bool(api_key))

API Key Loaded: True


## **Initialize LLM**

In [3]:
llm = ChatOpenAI(
    model="meta-llama/llama-3-8b-instruct",
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key,
    temperature=0.3
)

print("LLM initialized")

LLM initialized


## **Initialize Conversation Memory**

In [4]:
memory = ConversationBufferMemory()

conversation = ConversationChain(
    llm=llm,
    memory=memory,
    verbose=False
)

print("Conversation system ready")

Conversation system ready


C:\Users\ShoaibMohdKhan\AppData\Local\Temp\ipykernel_25272\3794717017.py:1: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()
C:\Users\ShoaibMohdKhan\AppData\Local\Temp\ipykernel_25272\3794717017.py:3: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. Build a conversational agent with `langchain.agents.create_agent` and persist message history via a LangGraph checkpointer.
  conversation = ConversationChain(


## **Reservation Assistant Prompt**

In [5]:
system_prompt = """
You are a restaurant reservation assistant.

Collect:
- customer name
- reservation date
- reservation time
- number of guests

Ask follow-up questions if details are missing.
"""

## **Start User Interaction**

In [6]:
user_message = input("Customer: ")

full_prompt = system_prompt + "\nUser: " + user_message

response = conversation.predict(input=full_prompt)

print("\nAgent 1:")
print(response)


Agent 1:
Welcome to our restaurant! I'd be happy to help you with your reservation. To confirm, you'd like a table for tomorrow, correct? Can you please tell me your name, so I can associate it with your reservation?

Also, what time would you like to dine with us tomorrow? Would you prefer an early lunch, a late lunch, or an evening dinner?


In [7]:
follow_up = input("\nCustomer Follow-up: ")

response = conversation.predict(input=follow_up)

print("\nAgent 1:")
print(response)


Agent 1:
I've taken note of your name, Shoaib, and the number of guests, 4. 

For the reservation date, I've confirmed that you'd like to dine with us tomorrow. And you've specified the time as 8 PM, which is a great choice. We have a lovely atmosphere in the evenings.

To finalize your reservation, I just need to confirm the date again. Just to make sure, tomorrow's date is March 22nd, correct?


## **Extract Structured Reservation Data**

In [8]:
conversation_history = memory.buffer

extract_prompt = f"""
Extract reservation details from this conversation.

Return ONLY valid JSON.

Conversation:
{conversation_history}

Required format:
{{
    "name": "",
    "date": "",
    "time": "",
    "guests": 0
}}
"""

response = llm.invoke(extract_prompt)

print(response.content)

Here is the extracted reservation details in JSON format:

```
{
    "name": "Shoaib",
    "date": "March 22nd",
    "time": "8 PM",
    "guests": 4
}
```


## **Clean JSON Response**

In [10]:
import json
import re

# Raw LLM output
raw_response = response.content

print("RAW RESPONSE:")
print(raw_response)

# Extract JSON block safely
json_match = re.search(r'\{[\s\S]*\}', raw_response)

if json_match:
    
    json_text = json_match.group()

    # Remove markdown wrappers if present
    json_text = json_text.replace("```json", "")
    json_text = json_text.replace("```", "")

    # Convert single quotes to double quotes
    json_text = json_text.replace("'", '"')

    print("\nEXTRACTED JSON:")
    print(json_text)

    try:
        reservation_data = json.loads(json_text)

        print("\nPARSED RESERVATION DATA:")
        print(reservation_data)

    except json.JSONDecodeError as e:
        print("\nJSON Parsing Error:")
        print(e)

else:
    print("No JSON object found in response.")

RAW RESPONSE:
Here is the extracted reservation details in JSON format:

```
{
    "name": "Shoaib",
    "date": "March 22nd",
    "time": "8 PM",
    "guests": 4
}
```

EXTRACTED JSON:
{
    "name": "Shoaib",
    "date": "March 22nd",
    "time": "8 PM",
    "guests": 4
}

PARSED RESERVATION DATA:
{'name': 'Shoaib', 'date': 'March 22nd', 'time': '8 PM', 'guests': 4}


## **Create Admin Request Function**

In [11]:
def generate_admin_request(reservation):
    
    message = f"""
    NEW RESERVATION REQUEST
    
    Customer Name : {reservation['name']}
    Reservation Date : {reservation['date']}
    Reservation Time : {reservation['time']}
    Number of Guests : {reservation['guests']}
    
    Approve or Reject?
    """
    
    return message

## **Generate Admin Request**

In [12]:
admin_message = generate_admin_request(reservation_data)

print(admin_message)


    NEW RESERVATION REQUEST

    Customer Name : Shoaib
    Reservation Date : March 22nd
    Reservation Time : 8 PM
    Number of Guests : 4

    Approve or Reject?
    


In [13]:
admin_response = input("Enter admin decision (approve/reject): ")

print("Admin Decision:", admin_response)

Admin Decision: approve


## **Process Admin Decision**

In [14]:
def process_admin_decision(decision):
    
    decision = decision.strip().lower()
    
    if decision == "approve":
        return {
            "status": "approved",
            "message": "Reservation approved by administrator."
        }
    
    elif decision == "reject":
        return {
            "status": "rejected",
            "message": "Reservation rejected by administrator."
        }
    
    else:
        return {
            "status": "invalid",
            "message": "Invalid admin response."
        }

## **Generate Final Result**

In [16]:
from datetime import datetime

final_result = process_admin_decision(admin_response)

final_result["processed_at"] = str(datetime.now())

print(final_result)

{'status': 'approved', 'message': 'Reservation approved by administrator.', 'processed_at': '2026-05-24 19:04:31.447325'}


## **Final User Notification**


In [17]:
if final_result["status"] == "approved":
    
    user_notification = f"""
    Dear {reservation_data['name']},
    
    Your reservation for {reservation_data['guests']} guests
    on {reservation_data['date']} at {reservation_data['time']}
    has been APPROVED.
    """
    
elif final_result["status"] == "rejected":
    
    user_notification = f"""
    Dear {reservation_data['name']},
    
    We regret to inform you that your reservation request
    has been REJECTED.
    """
    
else:
    
    user_notification = "Invalid administrator response."

print(user_notification)


    Dear Shoaib,

    Your reservation for 4 guests
    on March 22nd at 8 PM
    has been APPROVED.
    


## **Complete Workflow Summary**

In [18]:
print("===== FULL HUMAN-IN-THE-LOOP WORKFLOW =====")
print()

print("1. User Request:")
print(user_message)

print("\n2. Structured Reservation:")
print(reservation_data)

print("\n3. Admin Request:")
print(admin_message)

print("\n4. Final Decision:")
print(final_result)

print("\n5. User Notification:")
print(user_notification)

===== FULL HUMAN-IN-THE-LOOP WORKFLOW =====

1. User Request:
I want a table tommorow

2. Structured Reservation:
{'name': 'Shoaib', 'date': 'March 22nd', 'time': '8 PM', 'guests': 4}

3. Admin Request:

    NEW RESERVATION REQUEST

    Customer Name : Shoaib
    Reservation Date : March 22nd
    Reservation Time : 8 PM
    Number of Guests : 4

    Approve or Reject?
    

4. Final Decision:
{'status': 'approved', 'message': 'Reservation approved by administrator.', 'processed_at': '2026-05-24 19:04:31.447325'}

5. User Notification:

    Dear Shoaib,

    Your reservation for 4 guests
    on March 22nd at 8 PM
    has been APPROVED.
    
